In [1]:
import os
import glob
from dotenv import load_dotenv

from langchain_chroma import Chroma
from langchain_huggingface import HuggingFaceEmbeddings
from langchain_community.document_loaders import DirectoryLoader, TextLoader
from langchain_text_splitters import RecursiveCharacterTextSplitter

In [2]:
MODEL = "gpt-4.1-nano"
db_name = "vector_db"

load_dotenv(override = True)

True

In [3]:
folders = glob.glob("knowledge-base/*")

documents = []

for folder in folders:
    doc_type = os.path.basename(folder)

    loader = DirectoryLoader(
        folder,
        glob = "**/*.md",
        loader_cls = TextLoader,
        loader_kwargs = {"encoding": "utf-8"}
    )

    folder_docs = loader.load()

    for doc in folder_docs:
        doc.metadata["doc_type"] = doc_type
        documents.append(doc)

print(f"Loaded {len(documents)} documents")

Loaded 76 documents


In [4]:
text_splitter = RecursiveCharacterTextSplitter(
    chunk_size = 1000,
    chunk_overlap = 200
)

chunks = text_splitter.split_documents(documents)

print(f"Divided into {len(chunks)} chunks")
print(f"First chunk:\n\n{chunks[0]}")

Divided into 413 chunks
First chunk:

page_content='# About Insurellm

Insurellm was founded by Avery Lancaster in 2015 as an insurance tech startup designed to disrupt an industry in need of innovative products. Its first product was Markellm, the marketplace connecting consumers with insurance providers.

The company experienced rapid growth in its first five years, expanding its product portfolio to include Carllm (auto insurance portal), Homellm (home insurance portal), and Rellm (enterprise reinsurance platform). By 2020, Insurellm had reached a peak of 200 employees with 12 offices across the US.' metadata={'source': 'knowledge-base\\company\\about.md', 'doc_type': 'company'}


In [ ]:
embeddings = HuggingFaceEmbeddings(
    model_name = "all-MiniLM-L6-v2"
)

if os.path.exists(db_name):
    Chroma(
        persist_directory = db_name,
        embedding_function = embeddings
    ).delete_collection()

vectorstore = Chroma.from_documents(
    documents = chunks,
    embedding = embeddings,
    persist_directory = db_name
)

print(
    f"Vectorstore created with "
    f"{vectorstore._collection.count()} documents"
)

Vectorstore created with 413 documents


In [6]:
collection = vectorstore._collection
count = collection.count()

sample_embedding = collection.get(
    limit = 1,
    include = ["embeddings"]
)["embeddings"][0]

dimensions = len(sample_embedding)

print(
    f"There are {count:,} vectors with "
    f"{dimensions:,} dimensions in the vector store"
)

There are 413 vectors with 384 dimensions in the vector store


In [7]:
from langchain_openai import ChatOpenAI
from langchain_core.messages import SystemMessage, HumanMessage
import gradio as gr

In [8]:
retriever = vectorstore.as_retriever()

llm = ChatOpenAI(
    temperature = 0,
    model_name = MODEL
)

In [9]:
retriever.invoke("Who is Avery?")

[Document(id='61a0c853-3f65-40dd-9460-47a015a52fc4', metadata={'doc_type': 'employees', 'source': 'knowledge-base\\employees\\Avery Lancaster.md'}, page_content="## Other HR Notes\n- **Professional Development**: Avery has actively participated in leadership training programs and industry conferences, representing Insurellm and fostering partnerships.  \n- **Diversity & Inclusion Initiatives**: Avery has championed a commitment to diversity in hiring practices, seeing visible improvements in team representation since 2021.  \n- **Work-Life Balance**: Feedback revealed concerns regarding work-life balance, which Avery has approached by implementing flexible working conditions and ensuring regular check-ins with the team.\n- **Community Engagement**: Avery led community outreach efforts, focusing on financial literacy programs, particularly aimed at underserved populations, improving Insurellm's corporate social responsibility image.  \n\nAvery Lancaster has demonstrated resilience and a

In [10]:
SYSTEM_PROMPT_TEMPLATE = """
You are a knowledgeable, friendly assistant representing the company Insurellm.
You are chatting with a user about Insurellm.
If relevant, use the given context to answer any question.
If you don't know the answer, say so.

Context:
{context}
"""

In [11]:
def answer_question(question: str, history):
    docs = retriever.invoke(question)

    context = "\n\n".join(
        f"Source: {doc.metadata['source']}\n{doc.page_content}"
        for doc in docs
    )

    system_prompt = SYSTEM_PROMPT_TEMPLATE.format(
        context=context
    )

    response = llm.invoke([
        SystemMessage(content = system_prompt),
        HumanMessage(content = question)
    ])

    return response.content

In [12]:
answer_question(
    "Who is Avery Lancaster?",
    []
)

'Avery Lancaster is the Co-Founder and Chief Executive Officer (CEO) of Insurellm. She was born on March 15, 1985, and is based in San Francisco, California. Avery has been with Insurellm since 2015, guiding the company to become a leading player in the insurance technology industry. She has a background as a Senior Product Manager at Innovate Insurance Solutions before co-founding Insurellm. Avery is known for her innovative leadership, risk management expertise, and her active involvement in professional development, diversity initiatives, and community outreach.'

In [13]:
gr.ChatInterface(
    answer_question,
    type = "messages"
).launch()

* Running on local URL:  http://127.0.0.1:7860
* To create a public link, set `share=True` in `launch()`.


In [14]:
from evaluation import test
from collections import Counter

tests = test.load_tests()

print(f"Number of tests: {len(tests)}")

Number of tests: 150


In [15]:
example = tests[0]

print("Question:", example.question)
print("Category:", example.category)
print("Reference answer:", example.reference_answer)
print("Keywords:", example.keywords)

Question: Who won the prestigious IIOTY award in 2023?
Category: direct_fact
Reference answer: Maxine Thompson won the prestigious Insurellm Innovator of the Year (IIOTY) award in 2023.
Keywords: ['Maxine', 'Thompson', 'IIOTY']


In [16]:
Counter(t.category for t in tests)

Counter({'direct_fact': 70,
         'temporal': 20,
         'spanning': 20,
         'comparative': 10,
         'numerical': 10,
         'relationship': 10,
         'holistic': 10})

In [17]:
def evaluate_retrieval(test_case):
    docs = retriever.invoke(test_case.question)

    retrieved_text = "\n\n".join(
        doc.page_content for doc in docs
    ).lower()

    found_keywords = [
        keyword
        for keyword in test_case.keywords
        if keyword.lower() in retrieved_text
    ]

    keywords_found = len(found_keywords)
    total_keywords = len(test_case.keywords)

    keyword_coverage = (
        keywords_found / total_keywords * 100
        if total_keywords > 0
        else 0
    )

    return {
        "keywords_found": keywords_found,
        "total_keywords": total_keywords,
        "keyword_coverage": keyword_coverage,
        "found_keywords": found_keywords,
        "docs": docs
    }

In [18]:
retrieval_eval = evaluate_retrieval(example)

print("Keywords found:", retrieval_eval["keywords_found"])
print("Total keywords:", retrieval_eval["total_keywords"])
print("Keyword coverage:", retrieval_eval["keyword_coverage"])
print("Found:", retrieval_eval["found_keywords"])

Keywords found: 2
Total keywords: 3
Keyword coverage: 66.66666666666666
Found: ['Maxine', 'IIOTY']


In [19]:
from pydantic import BaseModel, Field

In [20]:
class AnswerEval(BaseModel):
    feedback: str = Field(
        description = "Brief explanation of the evaluation"
    )
    accuracy: float = Field(
        description = "Accuracy score from 1 to 5"
    )
    completeness: float = Field(
        description = "Completeness score from 1 to 5"
    )
    relevance: float = Field(
        description = "Relevance score from 1 to 5"
    )

In [21]:
evaluator_llm = llm.with_structured_output(AnswerEval)

In [22]:
def evaluate_answer(test_case):
    answer = answer_question(
        test_case.question,
        []
    )

    evaluation_prompt = f"""
You are evaluating the output of a RAG question-answering system.

Question:
{test_case.question}

Reference answer:
{test_case.reference_answer}

RAG answer:
{answer}

Evaluate the RAG answer on three criteria:

1. Accuracy: Is the answer factually correct compared with the reference answer?
2. Completeness: Does it include the important information from the reference answer?
3. Relevance: Does it directly answer the question without irrelevant information?

Give each criterion a score from 1 to 5.

Also provide a brief explanation.
"""

    evaluation = evaluator_llm.invoke(evaluation_prompt)

    return evaluation, answer

In [23]:
evaluation, answer = evaluate_answer(example)

print("Question:")
print(example.question)

print("\nReference answer:")
print(example.reference_answer)

print("\nRAG answer:")
print(answer)

print("\nEvaluation:")
print(evaluation)

Question:
Who won the prestigious IIOTY award in 2023?

Reference answer:
Maxine Thompson won the prestigious Insurellm Innovator of the Year (IIOTY) award in 2023.

RAG answer:
Maxine Thompson was recognized as the Insurellm Innovator of the Year (IIOTY) in 2023.

Evaluation:
feedback="The RAG answer correctly identifies Maxine Thompson as the winner and mentions the award name and year, aligning with the reference. However, it omits the word 'won,' which slightly affects the clarity of the achievement." accuracy=4.0 completeness=4.0 relevance=5.0


In [24]:
print("Feedback:", evaluation.feedback)
print("Accuracy:", evaluation.accuracy)
print("Completeness:", evaluation.completeness)
print("Relevance:", evaluation.relevance)

Feedback: The RAG answer correctly identifies Maxine Thompson as the winner and mentions the award name and year, aligning with the reference. However, it omits the word 'won,' which slightly affects the clarity of the achievement.
Accuracy: 4.0
Completeness: 4.0
Relevance: 5.0
